# Sequential Release Readiness Pipeline

| Field | Value |
| --- | --- |
| Scenario id | `sequential-release-readiness` |
| Pattern | `sequential` |
| API | `Responses API` |

**Learning goal:** Learn how Responses API clients can send a normal chat request while the server runs a fixed multi-stage agent pipeline.

> Use Responses plus sequential orchestration when each turn should produce a conversational answer through a predictable chain of review stages.


In [ ]:
import os

from IPython.display import HTML, display


_APTOS_STYLE = """
<style>
:root { --jp-content-font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif; }
.jp-RenderedHTMLCommon, .jp-RenderedMarkdown, .rendered_html, .jp-OutputArea-output {
    font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif;
    line-height: 1.55;
}
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3 {
    font-family: 'Aptos Display', 'Aptos', 'Segoe UI', sans-serif;
    font-weight: 600;
}
</style>
"""


def apply_notebook_style() -> str:
    display(HTML(_APTOS_STYLE))
    return _APTOS_STYLE


OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:14b")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MCP_TOOL_FUNCTIONS: dict[str, object] = {}

apply_notebook_style()
print(f"Ollama target: {OLLAMA_HOST} / {OLLAMA_MODEL}")


## Pattern: Sequential

Sequential orchestration is a fixed pipeline. Each agent receives the original request plus the work accumulated so far, so the output should read like a controlled handoff from stage to stage.

Best fit: repeatable workflows where every request needs the same ordered checks.

## API Shape

**Responses API -- startup-selected scenario shape.** The scenario and orchestration pattern are wired in at server start. Each client request uses the standard OpenAI-compatible `/responses` body -- a plain chat-style input. The client never specifies which agents run; the server owns the orchestration entirely.

This is a starter scenario. The domain is intentionally lightweight so the orchestration mechanics are easy to trace before the enterprise and quote-to-cash notebooks layer in MCP tool calls and richer context.

## Pattern Anatomy

| Dimension | Detail |
| --- | --- |
| Control flow | Agents run in a fixed order, with each stage receiving the prior stage response. |
| Coordination | The graph defines the chain. The model does not decide which agent acts next. |
| Output | The terminal output includes the stage transcript. |
| Best when | Use for repeatable pipelines where every request needs the same stages. |

## Instruction-Led LLM Agents

| Agent | Role | Capabilities |
| --- | --- | --- |
| `ScopePlannerAgent` | Extracts release scope and readiness questions. | Instructions only |
| `DependencyPlannerAgent` | Identifies dependencies and sequencing. | Instructions only |
| `RiskReviewerAgent` | Reviews operational and customer risk. | Instructions only |
| `DocsWriterAgent` | Plans release notes and internal docs. | Instructions only |
| `FinalEditorAgent` | Creates the final readiness brief. | Instructions only |

> **Instructor note:** Each row is an LLM-backed agent with role instructions.
> Most agents rely on instructions alone; enterprise and quote-to-cash agents may
> also call domain tools for grounded context.


In [ ]:
from dataclasses import dataclass
from typing import Any

from agent_framework.ollama import OllamaChatClient


DEFAULT_OLLAMA_TEMPERATURE = 0.0
DEFAULT_OLLAMA_NUM_CTX = 8192
DEFAULT_OLLAMA_KEEP_ALIVE = "10m"
DEFAULT_OLLAMA_THINK = False

_UNSUPPORTED_OLLAMA_CHAT_OPTIONS = {
    "allow_multiple_tool_calls",
    "conversation_id",
    "logit_bias",
    "metadata",
    "store",
    "user",
}


@dataclass(frozen=True)
class OllamaAgentConfig:
    model: str
    host: str
    temperature: float
    num_ctx: int
    max_tokens: int
    keep_alive: str
    think: bool

    def default_options(self) -> dict[str, Any]:
        return {
            "temperature": self.temperature,
            "num_ctx": self.num_ctx,
            "max_tokens": self.max_tokens,
            "keep_alive": self.keep_alive,
            "think": self.think,
        }


def parse_env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be true or false.")


def build_ollama_config(
    *,
    model: str | None = None,
    host: str | None = None,
    temperature: float | None = None,
    num_ctx: int | None = None,
    max_tokens: int | None = None,
    keep_alive: str | None = None,
    think: bool | None = None,
) -> OllamaAgentConfig:
    return OllamaAgentConfig(
        model=model or os.getenv("OLLAMA_MODEL") or "qwen3:14b",
        host=host or os.getenv("OLLAMA_HOST") or "http://localhost:11434",
        temperature=temperature
        if temperature is not None
        else float(os.getenv("OLLAMA_TEMPERATURE", str(DEFAULT_OLLAMA_TEMPERATURE))),
        num_ctx=num_ctx if num_ctx is not None else int(os.getenv("OLLAMA_NUM_CTX", str(DEFAULT_OLLAMA_NUM_CTX))),
        max_tokens=max_tokens if max_tokens is not None else int(os.getenv("OLLAMA_MAX_TOKENS", "500")),
        keep_alive=keep_alive or os.getenv("OLLAMA_KEEP_ALIVE") or DEFAULT_OLLAMA_KEEP_ALIVE,
        think=think if think is not None else parse_env_bool("OLLAMA_THINK", DEFAULT_OLLAMA_THINK),
    )


class ScenarioOllamaChatClient(OllamaChatClient):
    def _prepare_options(self, messages: Any, options: Any) -> dict[str, Any]:
        filtered = {
            key: value
            for key, value in dict(options).items()
            if key not in _UNSUPPORTED_OLLAMA_CHAT_OPTIONS
        }
        return super()._prepare_options(messages, filtered)


def make_agent(spec: Any, *, config: OllamaAgentConfig | None = None) -> Any:
    resolved = config or build_ollama_config()
    instructions = f"You are {spec.name}. {spec.instructions}"
    tools = tools_for_agent(spec)
    return ScenarioOllamaChatClient(host=resolved.host, model=resolved.model).as_agent(
        name=spec.name,
        description=spec.description,
        instructions=instructions,
        tools=tools or None,
        default_options=resolved.default_options(),
        require_per_service_call_history_persistence=True,
    )


In [ ]:
import json
from dataclasses import dataclass
from typing import Any, Sequence


@dataclass(frozen=True)
class AgentSpec:
    name: str
    description: str
    instructions: str
    mcp_tools: tuple[str, ...] = ()
    mcp_server: str = "enterprise_context"
    route_keywords: tuple[str, ...] = ()


@dataclass(frozen=True)
class ScenarioSpec:
    id: str
    pattern: str
    title: str
    learning_goal: str
    when_to_use: str
    sample_input: str
    agents: tuple[AgentSpec, ...]
    handoff_finisher: str | None = None
    concurrent_synthesizer: str | None = None


SCENARIO_DATA = json.loads(
    r"""
{
  "id": "sequential-release-readiness",
  "pattern": "sequential",
  "title": "Sequential Release Readiness Pipeline",
  "learning_goal": "Learn how Responses API clients can send a normal chat request while the server runs a fixed multi-stage agent pipeline.",
  "when_to_use": "Use Responses plus sequential orchestration when each turn should produce a conversational answer through a predictable chain of review stages.",
  "sample_input": "Prepare a release readiness brief for version 2.4.0 with billing reconciliation, dashboard exports, and API bug fixes.",
  "handoff_finisher": null,
  "concurrent_synthesizer": null,
  "agents": [
    {
      "name": "ScopePlannerAgent",
      "description": "Extracts release scope and readiness questions.",
      "instructions": "Turn the user request into a concrete release-scope summary and ordered readiness questions.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": []
    },
    {
      "name": "DependencyPlannerAgent",
      "description": "Identifies dependencies and sequencing.",
      "instructions": "Identify upstream/downstream teams, rollout dependencies, migration needs, and sequencing constraints.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": []
    },
    {
      "name": "RiskReviewerAgent",
      "description": "Reviews operational and customer risk.",
      "instructions": "Assess operational risk, rollback risk, security exposure, and customer impact. Return mitigations.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": []
    },
    {
      "name": "DocsWriterAgent",
      "description": "Plans release notes and internal docs.",
      "instructions": "Create release-note bullets, internal enablement tasks, and support-facing docs needs.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": []
    },
    {
      "name": "FinalEditorAgent",
      "description": "Creates the final readiness brief.",
      "instructions": "Synthesize the prior outputs into scope, risks, required follow-ups, and a go/no-go recommendation.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": []
    }
  ]
}
    """
)
AGENTS = tuple(
    AgentSpec(
        name=item["name"],
        description=item["description"],
        instructions=item["instructions"],
        mcp_tools=tuple(item.get("mcp_tools", [])),
        mcp_server=item.get("mcp_server", "enterprise_context"),
        route_keywords=tuple(item.get("route_keywords", [])),
    )
    for item in SCENARIO_DATA["agents"]
)
SCENARIO = ScenarioSpec(
    id=SCENARIO_DATA["id"],
    pattern=SCENARIO_DATA["pattern"],
    title=SCENARIO_DATA["title"],
    learning_goal=SCENARIO_DATA["learning_goal"],
    when_to_use=SCENARIO_DATA["when_to_use"],
    sample_input=SCENARIO_DATA["sample_input"],
    agents=AGENTS,
    handoff_finisher=SCENARIO_DATA.get("handoff_finisher"),
    concurrent_synthesizer=SCENARIO_DATA.get("concurrent_synthesizer"),
)


def tools_for_agent(spec: AgentSpec) -> list[object]:
    tools: list[object] = []
    for tool_name in spec.mcp_tools:
        try:
            tools.append(MCP_TOOL_FUNCTIONS[tool_name])
        except KeyError as exc:
            raise ValueError(f"Missing inlined tool '{tool_name}' for {spec.name}.") from exc
    return tools


def scenario_summary(scenario: ScenarioSpec) -> dict[str, str]:
    return {
        "id": scenario.id,
        "title": scenario.title,
        "pattern": scenario.pattern,
        "learning_goal": scenario.learning_goal,
        "when_to_use": scenario.when_to_use,
        "sample": getattr(scenario, "sample_input"),
    }


def agent_capability_map(scenario: ScenarioSpec) -> list[dict[str, Any]]:
    return [
        {
            "agent": spec.name,
            "description": spec.description,
            "instructions": spec.instructions,
            "mcp_tools": list(spec.mcp_tools),
            "mcp_server": spec.mcp_server if spec.mcp_tools else None,
        }
        for spec in scenario.agents
    ]


def mcp_tool_context(scenario: ScenarioSpec) -> dict[str, Any]:
    tools_by_agent = {spec.name: list(spec.mcp_tools) for spec in scenario.agents if spec.mcp_tools}
    all_tools_used = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    return {
        "uses_mcp": bool(all_tools_used),
        "tools_by_agent": tools_by_agent,
        "all_tools_used": all_tools_used,
    }


MAX_TOKENS = 500 if SCENARIO.pattern == "magentic" else 250
RESPONSES_PAYLOAD = {"input": SCENARIO.sample_input, "stream": False}
SAMPLE_PROMPT = SCENARIO.sample_input

print(json.dumps(scenario_summary(SCENARIO), indent=2))
print(json.dumps(agent_capability_map(SCENARIO), indent=2))
if mcp_tool_context(SCENARIO)["uses_mcp"]:
    print(json.dumps(mcp_tool_context(SCENARIO), indent=2))


In [ ]:
import re
from collections.abc import Mapping
from typing import Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)


_TRANSCRIPT_KEY = "transcript"
_STOPWORDS = {"agent", "specialist", "the", "and", "for", "with", "that", "from", "into"}


def make_request(text: str) -> AgentExecutorRequest:
    return AgentExecutorRequest(messages=[Message(role="user", contents=[text])])


def response_text(response: AgentExecutorResponse) -> str:
    agent_response = getattr(response, "agent_response", None)
    return (getattr(agent_response, "text", None) or "").strip()


def _append_transcript(ctx: WorkflowContext[Any], author: str, text: str) -> list[str]:
    transcript = list(ctx.get_state(_TRANSCRIPT_KEY) or [])
    transcript.append(f"[{author}] {text}")
    ctx.set_state(_TRANSCRIPT_KEY, transcript)
    return transcript


class PromptDispatchExecutor(Executor):
    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        ctx.set_state("prompt", prompt)
        ctx.set_state(_TRANSCRIPT_KEY, [])
        await ctx.send_message(make_request(prompt))


class StageGateExecutor(Executor):
    def __init__(self, id: str, *, stage_name: str) -> None:
        super().__init__(id=id)
        self._stage_name = stage_name

    @handler
    async def gate(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        transcript = _append_transcript(ctx, self._stage_name, response_text(response))
        prompt = ctx.get_state("prompt") or ""
        carried = "\n".join(transcript)
        await ctx.send_message(make_request(f"Original request:\n{prompt}\n\nWork so far:\n{carried}"))


class SequentialOutputExecutor(Executor):
    def __init__(self, id: str, *, stage_name: str) -> None:
        super().__init__(id=id)
        self._stage_name = stage_name

    @handler
    async def finish(self, response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
        transcript = _append_transcript(ctx, self._stage_name, response_text(response))
        await ctx.yield_output("\n\n".join(transcript))


class ConcurrentAggregatorExecutor(Executor):
    def __init__(self, id: str, *, agent_names: list[str]) -> None:
        super().__init__(id=id)
        self._agent_names = agent_names

    @handler
    async def aggregate(self, responses: list[AgentExecutorResponse], ctx: WorkflowContext[Never, str]) -> None:
        labelled: list[str] = []
        for index, response in enumerate(responses):
            name = self._agent_names[index] if index < len(self._agent_names) else f"agent{index + 1}"
            labelled.append(f"[{name}]\n{response_text(response)}")
        await ctx.yield_output("\n\n".join(labelled))


class ConcurrentSynthesisGateExecutor(Executor):
    def __init__(self, id: str, *, agent_names: list[str]) -> None:
        super().__init__(id=id)
        self._agent_names = agent_names

    @handler
    async def gate(self, responses: list[AgentExecutorResponse], ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        for index, response in enumerate(responses):
            name = self._agent_names[index] if index < len(self._agent_names) else f"agent{index + 1}"
            _append_transcript(ctx, name, response_text(response))
        prompt = ctx.get_state("prompt") or ""
        carried = "\n".join(ctx.get_state(_TRANSCRIPT_KEY) or [])
        await ctx.send_message(
            make_request(
                f"You are the synthesis stage.\nOriginal request:\n{prompt}\n\n"
                f"Independent specialist findings:\n{carried}\n\n"
                "Combine these findings into the final deliverable."
            )
        )


_ROUTE_DIRECTIVE = re.compile(r"route\s*:\s*([A-Za-z][A-Za-z0-9 _-]*)", re.IGNORECASE)


def _route_slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


class HandoffRouterExecutor(Executor):
    def __init__(
        self,
        id: str,
        *,
        routes: dict[str, tuple[str, ...]],
        default_route: str,
        display_names: dict[str, str] | None = None,
    ) -> None:
        super().__init__(id=id)
        self._routes = routes
        self._default_route = default_route
        self._display_names = display_names or {}

    def directed(self, text: str) -> str | None:
        for match in reversed(_ROUTE_DIRECTIVE.findall(text)):
            slug = _route_slug(match)
            if slug in self._routes:
                return slug
        return None

    def choose(self, text: str) -> str:
        directed = self.directed(text)
        if directed is not None:
            return directed
        lowered = text.lower()
        best_route, best_hits = self._default_route, 0
        for route, keywords in self._routes.items():
            hits = sum(1 for keyword in keywords if keyword in lowered)
            if hits > best_hits:
                best_route, best_hits = route, hits
        return best_route

    @handler
    async def route(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        triage_text = response_text(response)
        chosen = self.choose(triage_text)
        ctx.set_state("route", chosen)
        ctx.set_state("route_name", self._display_names.get(chosen, chosen))
        ctx.set_state("route_source", "model-directive" if self.directed(triage_text) else "keyword-score")
        prompt = ctx.get_state("prompt") or ""
        await ctx.send_message(
            make_request(f"Triage routed this to you.\nRequest:\n{prompt}\n\nTriage notes:\n{triage_text}"),
            target_id=chosen,
        )


class HandoffFinisherGateExecutor(Executor):
    @handler
    async def gate(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        route = ctx.get_state("route_name") or ctx.get_state("route") or "specialist"
        transcript = _append_transcript(ctx, route, response_text(response))
        prompt = ctx.get_state("prompt") or ""
        carried = "\n".join(transcript)
        await ctx.send_message(
            make_request(
                f"You are the finishing stage of a handoff.\nOriginal request:\n{prompt}\n\n"
                f"Routed specialist notes:\n{carried}\n\nComplete the final deliverable."
            )
        )


class HandoffOutputExecutor(Executor):
    def __init__(self, id: str, *, stage_name: str | None = None) -> None:
        super().__init__(id=id)
        self._stage_name = stage_name

    @handler
    async def finish(self, response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
        route = ctx.get_state("route_name") or ctx.get_state("route") or "specialist"
        source = ctx.get_state("route_source") or "keyword-score"
        header = f"[routed to {route} via {source}]"
        if self._stage_name is None:
            await ctx.yield_output(f"{header}\n{response_text(response)}")
            return
        transcript = _append_transcript(ctx, self._stage_name, response_text(response))
        await ctx.yield_output("\n\n".join([header, *transcript]))


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def _agents_for(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> list[Any]:
    return [make_agent(spec, config=config) for spec in scenario.agents]


def _agent_executor(spec_index: int, scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> AgentExecutor:
    spec = scenario.agents[spec_index]
    return AgentExecutor(make_agent(spec, config=config), id=_slug(spec.name))


def _route_keywords(spec: AgentSpec) -> tuple[str, ...]:
    if spec.route_keywords:
        return tuple(spec.route_keywords)
    tokens = re.findall(r"[a-z]+", f"{spec.name} {spec.description}".lower())
    keywords = [token for token in tokens if len(token) > 3 and token not in _STOPWORDS]
    return tuple(dict.fromkeys(keywords))[:6]


def build_sequential_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    agents = [_agent_executor(i, scenario, config=config) for i in range(len(scenario.agents))]
    dispatch = PromptDispatchExecutor(id="dispatch")
    output = SequentialOutputExecutor(id="final_output", stage_name=scenario.agents[-1].name)
    builder = WorkflowBuilder(start_executor=dispatch, output_from=[output])
    builder.add_edge(dispatch, agents[0])
    for index in range(len(agents) - 1):
        gate = StageGateExecutor(id=f"gate_{index}", stage_name=scenario.agents[index].name)
        builder.add_edge(agents[index], gate)
        builder.add_edge(gate, agents[index + 1])
    builder.add_edge(agents[-1], output)
    return builder.build()


def build_concurrent_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    synthesizer_name = scenario.concurrent_synthesizer
    parallel = [i for i in range(len(scenario.agents)) if scenario.agents[i].name != synthesizer_name]
    agents = [_agent_executor(i, scenario, config=config) for i in parallel]
    parallel_names = [scenario.agents[i].name for i in parallel]
    dispatch = PromptDispatchExecutor(id="dispatch")
    if synthesizer_name is None:
        aggregator = ConcurrentAggregatorExecutor(id="aggregator", agent_names=parallel_names)
        builder = WorkflowBuilder(start_executor=dispatch, output_from=[aggregator])
        builder.add_fan_out_edges(dispatch, agents)
        builder.add_fan_in_edges(agents, aggregator)
        return builder.build()
    synthesizer_index = next(
        i for i in range(len(scenario.agents)) if scenario.agents[i].name == synthesizer_name
    )
    synthesizer = _agent_executor(synthesizer_index, scenario, config=config)
    gate = ConcurrentSynthesisGateExecutor(id="synthesis_gate", agent_names=parallel_names)
    output = SequentialOutputExecutor(id="final_output", stage_name=synthesizer_name)
    builder = WorkflowBuilder(start_executor=dispatch, output_from=[output])
    builder.add_fan_out_edges(dispatch, agents)
    builder.add_fan_in_edges(agents, gate)
    builder.add_edge(gate, synthesizer)
    builder.add_edge(synthesizer, output)
    return builder.build()


def build_handoff_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    triage = _agent_executor(0, scenario, config=config)
    finisher_name = scenario.handoff_finisher
    routable = [i for i in range(1, len(scenario.agents)) if scenario.agents[i].name != finisher_name]
    specialists = [_agent_executor(i, scenario, config=config) for i in routable]
    specialist_ids = [_slug(scenario.agents[i].name) for i in routable]
    routes = {specialist_ids[pos]: _route_keywords(scenario.agents[i]) for pos, i in enumerate(routable)}
    display_names = {specialist_ids[pos]: scenario.agents[i].name for pos, i in enumerate(routable)}
    dispatch = PromptDispatchExecutor(id="dispatch")
    router = HandoffRouterExecutor(
        id="router", routes=routes, default_route=specialist_ids[0], display_names=display_names
    )
    output = HandoffOutputExecutor(id="final_output", stage_name=finisher_name)
    builder = WorkflowBuilder(start_executor=dispatch, output_from=[output])
    builder.add_edge(dispatch, triage)
    builder.add_edge(triage, router)
    if finisher_name is None:
        for specialist in specialists:
            builder.add_edge(router, specialist)
            builder.add_edge(specialist, output)
        return builder.build()
    finisher_index = next(
        i for i in range(1, len(scenario.agents)) if scenario.agents[i].name == finisher_name
    )
    finisher = _agent_executor(finisher_index, scenario, config=config)
    finisher_gate = HandoffFinisherGateExecutor(id="finisher_gate")
    for specialist in specialists:
        builder.add_edge(router, specialist)
        builder.add_edge(specialist, finisher_gate)
    builder.add_edge(finisher_gate, finisher)
    builder.add_edge(finisher, output)
    return builder.build()


def build_group_chat_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    from agent_framework.orchestrations import GroupChatBuilder

    participants = _agents_for(scenario, config=config)

    def round_robin_selector(state: Any) -> str:
        participant_names = list(state.participants.keys())
        return participant_names[state.current_round % len(participant_names)]

    def stop_after_council(messages: list[Any]) -> bool:
        assistant_messages = [m for m in messages if getattr(m, "role", None) == "assistant"]
        if len(assistant_messages) >= 7:
            return True
        last_text = getattr(messages[-1], "text", "").lower() if messages else ""
        return "approved" in last_text and "recommendation" in last_text

    return GroupChatBuilder(
        participants=participants,
        selection_func=round_robin_selector,
        termination_condition=stop_after_council,
        intermediate_output_from=participants,
    ).build()


def build_magentic_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    from agent_framework.orchestrations import MagenticBuilder

    agents = _agents_for(scenario, config=config)
    manager_agent = agents[0]
    participants = agents[1:]
    return MagenticBuilder(
        participants=participants,
        intermediate_output_from=participants,
        manager_agent=manager_agent,
        max_round_count=10,
        max_stall_count=3,
        max_reset_count=2,
    ).build()


def build_workflow(
    scenario: ScenarioSpec = SCENARIO,
    *,
    max_tokens: int | None = None,
    **config_overrides: Any,
) -> Any:
    config = build_ollama_config(max_tokens=max_tokens or MAX_TOKENS, **config_overrides)
    builders = {
        "sequential": build_sequential_workflow,
        "concurrent": build_concurrent_workflow,
        "handoff": build_handoff_workflow,
        "group-chat": build_group_chat_workflow,
        "magentic": build_magentic_workflow,
    }
    return builders[scenario.pattern](scenario, config=config)


def workflow_result_to_text(result: Any) -> str:
    outputs = result.get_outputs() if hasattr(result, "get_outputs") else result
    intermediate = result.get_intermediate_outputs() if hasattr(result, "get_intermediate_outputs") else []
    if not outputs:
        intermediate_text = join_readable_outputs(intermediate)
        return clean_workflow_text(intermediate_text) or "No workflow output was produced."
    output_text = join_readable_outputs(outputs)
    if intermediate and should_use_intermediate_outputs(output_text):
        intermediate_text = join_readable_outputs(intermediate)
        if intermediate_text:
            return clean_workflow_text(intermediate_text)
    return clean_workflow_text(output_text) or "No readable workflow text was produced."


def join_readable_outputs(outputs: Any) -> str:
    return "\n\n".join(text for output in outputs if (text := agent_response_to_text(output)))


def should_use_intermediate_outputs(output_text: str) -> bool:
    normalized = output_text.strip().lower()
    if not normalized:
        return True
    if len(normalized) >= 160:
        return False
    markers = (
        "termination condition",
        "maximum reset count",
        "maximum stall count",
        "workflow terminated",
        "group chat has reached its termination condition",
    )
    return any(marker in normalized for marker in markers)


def agent_response_to_text(value: Any) -> str:
    text = clean_workflow_text(extract_text(value))
    return text


def clean_workflow_text(text: str) -> str:
    """Remove leading framework status lines when useful scenario text follows."""

    lines = text.splitlines()
    while lines and is_framework_status_line(lines[0]) and any(line.strip() for line in lines[1:]):
        lines.pop(0)
        while lines and not lines[0].strip():
            lines.pop(0)
    return "\n".join(lines).strip()


def is_framework_status_line(line: str) -> bool:
    normalized = line.strip().lower()
    return (
        normalized.startswith("invalid next speaker:")
        or normalized.startswith("magentic orchestrator:")
        or normalized.startswith("maximum consecutive function call errors reached")
    )


def extract_text(value: Any, seen: set[int] | None = None) -> str:
    if value is None:
        return ""
    if seen is None:
        seen = set()
    value_id = id(value)
    if value_id in seen:
        return ""
    seen.add(value_id)
    if isinstance(value, str):
        return "" if is_object_repr(value) else value
    text = getattr(value, "text", None)
    if isinstance(text, str) and text and not is_object_repr(text):
        return text
    messages = getattr(value, "messages", None)
    if messages:
        parts: list[str] = []
        for message in messages:
            author = getattr(message, "author_name", None) or getattr(message, "role", None) or "assistant"
            message_text = extract_text(message, seen)
            if message_text:
                parts.append(f"[{author}] {message_text}")
        if parts:
            return "\n".join(parts)
    contents = getattr(value, "contents", None)
    if contents:
        return "\n".join(part for content in contents if (part := extract_text(content, seen)))
    items = getattr(value, "items", None)
    if items and not callable(items):
        return "\n".join(part for item in items if (part := extract_text(item, seen)))
    result = getattr(value, "result", None)
    if result is not None:
        return extract_text(result, seen)
    if isinstance(value, Mapping):
        parts = [extract_text(value.get(key), seen) for key in ("text", "content", "message", "summary", "result")]
        return "\n".join(part for part in parts if part)
    if isinstance(value, Sequence) and not isinstance(value, (bytes, bytearray, str)):
        return "\n".join(part for item in value if (part := extract_text(item, seen)))
    fallback = str(value)
    return "" if is_object_repr(fallback) else fallback


def is_object_repr(value: str) -> bool:
    return value.startswith("<") and " object at 0x" in value and value.endswith(">")


def group_chat_learning_summary(
    scenario: ScenarioSpec,
    prompt: str,
    framework_text: str,
) -> str:
    """Explain a completed group-chat run when this framework build hides the transcript."""

    lines = [
        "Group chat completed.",
        "",
        f"Framework result: {framework_text.strip()}",
        "",
        "Learning view:",
        "- The workflow used Microsoft Agent Framework's GroupChatBuilder with LLM-backed participants.",
        "- Selection is code-defined round robin; termination is code-defined from assistant messages.",
        f"- The submitted scenario prompt was: {prompt}",
        "- Participant order:",
    ]
    for index, spec in enumerate(scenario.agents, start=1):
        tools = ", ".join(spec.mcp_tools) if spec.mcp_tools else "no domain tools"
        lines.append(f"  {index}. {spec.name}: {spec.description} ({tools})")
    tool_names = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    if tool_names:
        lines.append("- Grounding sources available to tool-enabled agents:")
        for tool_name in tool_names:
            lines.append(f"  - {tool_name}")
    lines.extend(
        [
            "",
            "Note: this local Agent Framework build returned the group-chat termination marker",
            "without exposing participant turns through get_intermediate_outputs(). The notebook",
            "keeps the framework run intact and prints this learning summary so the scenario",
            "still explains the orchestration shape and agent responsibilities.",
        ]
    )
    return "\n".join(lines)


## Flow Diagram

The diagram below shows a linear chain of 5 stages with a stage-gate executor between each pair attached to the Responses API /responses.
Solid arrows are orchestration edges. Dashed arrows (`-.->`) are tool calls.
Domain tool nodes use a stadium shape.


In [ ]:
import base64
import html
from dataclasses import dataclass

from IPython.display import HTML, display


@dataclass(frozen=True)
class ScenarioFlowDiagram:
    title: str
    mermaid: str
    image_url: str


def scenario_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _diagram_source(scenario, api_boundary="Responses API /responses", input_label="OpenAI-style input")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_scenario_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = scenario_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram


def _diagram_source(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    if scenario.pattern == "sequential":
        return _sequential_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    if scenario.pattern == "concurrent":
        return _concurrent_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    if scenario.pattern == "handoff":
        return _handoff_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    if scenario.pattern == "group-chat":
        return _group_chat_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    if scenario.pattern == "magentic":
        return _magentic_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    raise ValueError(f"Unsupported pattern '{scenario.pattern}'.")


def _sequential_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    previous = "orchestrator"
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(scenario.agents, start=1):
        node = f"agent{index}"
        lines.append(f"    {previous} -->|stage {index}| {node}[{_label(agent.name)}]")
        previous = node
        pairs.append((agent, node))
    lines.append(f"    {previous} --> output[Responses output]")
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)


def _concurrent_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    synthesizer = next(
        (agent for agent in scenario.agents if agent.name == scenario.concurrent_synthesizer), None
    )
    parallel = [agent for agent in scenario.agents if agent is not synthesizer]
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append("    orchestrator --> fanout{Fan out same request}")
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(parallel, start=1):
        node = f"agent{index}"
        lines.append(f"    fanout --> {node}[{_label(agent.name)}]")
        lines.append(f"    {node} --> aggregate{{Aggregate findings}}")
        pairs.append((agent, node))
    if synthesizer is None:
        lines.append("    aggregate --> output[Responses output]")
    else:
        lines.append(f"    aggregate --> synthesizer[{_label(synthesizer.name)}]")
        lines.append("    synthesizer --> output[Responses output]")
        pairs.append((synthesizer, "synthesizer"))
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)


def _handoff_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    triage, *others = scenario.agents
    finisher = next((agent for agent in others if agent.name == scenario.handoff_finisher), None)
    specialists = [agent for agent in others if agent is not finisher]
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append(f"    orchestrator --> triage[{_label(triage.name)}]")
    lines.append("    triage --> decision{Ownership decision}")
    pairs: list[tuple[AgentSpec, str]] = [(triage, "triage")]
    sink = "output[Responses output]"
    if finisher is not None:
        lines.append(f"    finisher[{_label(finisher.name)}] --> output[Responses output]")
        pairs.append((finisher, "finisher"))
        sink = "finisher"
    for index, agent in enumerate(specialists, start=1):
        node = f"specialist{index}"
        lines.append(f"    decision -->|handoff| {node}[{_label(agent.name)}]")
        lines.append(f"    {node} --> {sink}")
        pairs.append((agent, node))
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)


def _group_chat_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append("    orchestrator --> selector{Round-robin selector}")
    previous = "selector"
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(scenario.agents, start=1):
        node = f"agent{index}"
        lines.append(f"    {previous} --> {node}[{_label(agent.name)}]")
        previous = node
        pairs.append((agent, node))
    lines.append(f"    {previous} --> stop{{Termination condition}}")
    lines.append("    stop -->|continue| selector")
    lines.append("    stop -->|done| output[Responses output]")
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)


def _magentic_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    manager, *specialists = scenario.agents
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append(f"    orchestrator --> manager[{_label(manager.name)}]")
    lines.append("    manager --> plan{Plan and delegate}")
    pairs: list[tuple[AgentSpec, str]] = [(manager, "manager")]
    for index, agent in enumerate(specialists, start=1):
        node = f"specialist{index}"
        lines.append(f"    plan --> {node}[{_label(agent.name)}]")
        lines.append(f"    {node} --> progress{{Progress ledger}}")
        pairs.append((agent, node))
    lines.append("    progress -->|replan| manager")
    lines.append("    progress -->|complete or stop| output[Responses output]")
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)


def _header(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> list[str]:
    return [
        "flowchart TD",
        f"    client[{_label(input_label)}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
    ]


def _mcp_tool_links(pairs: list[tuple[AgentSpec, str]]) -> list[str]:
    lines: list[str] = []
    for agent, node_id in pairs:
        for tool in agent.mcp_tools:
            lines.append(f"    {node_id} -.->|mcp tool| tool_{tool}([{_label(tool)}])")
    return lines


def quote_to_cash_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _quote_to_cash_source(scenario, api_boundary="Responses API /responses")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Quote-To-Cash Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_quote_to_cash_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = quote_to_cash_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram


def _quote_to_cash_source(scenario: ScenarioSpec, *, api_boundary: str) -> str:
    names = {agent.name for agent in scenario.agents}

    def node(role: str) -> str:
        return role if role in names else next(iter(names))

    lines = [
        "flowchart TD",
        f"    client[{_label('Quote request begins in CRM')}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
        f"    orchestrator --> crm[{_label('CRM system')}]",
        f"    crm -->|wave 1| trigger[{_label(node('QuoteTriggerAgent'))}]",
        f"    crm -->|wave 1| customer[{_label(node('CustomerContextAgent'))}]",
        f"    trigger --> store1[({_label('Orchestration store: customer context')})]",
        "    customer --> store1",
        f"    store1 -. {_label('deallocate wave 1')} .-> freed1(({_label('agents released')}))",
        f"    store1 --> product[{_label('Product / SKU system')}]",
        f"    product -->|wave 2| sku[{_label(node('SkuDiscoveryAgent'))}]",
        f"    product -->|wave 2| fit[{_label(node('ProductFitAgent'))}]",
        f"    sku --> store2[({_label('Orchestration store: product context')})]",
        "    fit --> store2",
        f"    store2 -. {_label('deallocate wave 2')} .-> freed2(({_label('agents released')}))",
        f"    store2 --> pricingsys[{_label('Pricing / finance / legal system')}]",
        f"    pricingsys -->|wave 3| pricing[{_label(node('PricingTermsAgent'))}]",
        f"    pricing --> generation[{_label(node('QuoteGenerationAgent'))}]",
        f"    generation --> quote[/{_label('Final quote package')}/]",
    ]
    return "\n".join(lines)


def _label(value: str) -> str:
    return value.replace('"', "'")


def _mermaid_image_url(mermaid: str) -> str:
    encoded = base64.urlsafe_b64encode(mermaid.encode("utf-8")).decode("ascii").rstrip("=")
    return f"https://mermaid.ink/img/{encoded}"


flow_diagram = display_scenario_flow(SCENARIO)
print(flow_diagram.mermaid)


## Live Run

Each agent output is captured by a `StageGateExecutor` and appended to a growing transcript. The next agent receives both the original prompt and the accumulated work so far. The final cell prints the complete stage-by-stage log.

> **Instructor note:** `qwen3:14b` runs with `think: False` by default (extended reasoning off).
> Set `OLLAMA_THINK=true` before the environment cell to enable chain-of-thought reasoning --
> useful when debugging unexpected routing decisions or tool call sequences.


In [ ]:
import io
from contextlib import redirect_stderr, redirect_stdout


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
_framework_logs = io.StringIO()
with redirect_stdout(_framework_logs), redirect_stderr(_framework_logs):
    result = await workflow.run(SAMPLE_PROMPT)
framework_logs = _framework_logs.getvalue()
output_text = workflow_result_to_text(result)
if SCENARIO.pattern == "group-chat" and should_use_intermediate_outputs(output_text):
    output_text = group_chat_learning_summary(SCENARIO, SAMPLE_PROMPT, output_text)

if not output_text.strip():
    raise RuntimeError("Workflow completed but produced no readable text.")

print(output_text)


## What to Inspect

Compare the first stage output to the final editor output. Later stages should build on prior work, not repeat it -- each `StageGateExecutor` carries the full transcript forward. If a stage ignores prior context, inspect its instructions and the gate prompt to see exactly what the agent received.


## Experiments

- Change `RESPONSES_PAYLOAD['input']` and rerun the live cell.
- Override `OLLAMA_MODEL` or `OLLAMA_HOST` before the environment cell to target a different local Ollama setup.
- Inspect `agent_capability_map(SCENARIO)` and tighten one agent's instructions to see how orchestration behavior changes.
- Lower `MAX_TOKENS` for faster runs or raise it when sequential needs more room.
